# Mekaniske koblinger: Hvordan tegner man nesten en rett linje med roterende ledd?

## Watt-koblingen og moderne kinematikk

### Pilotprosjekt for Matematikk 1

På 1700- og 1800-tallet var det vanskelig å lage lange, nøyaktige glideskinner. Ingeniører utviklet derfor mekaniske koblinger som kunne føre et punkt langs en nesten rett linje ved hjelp av stive stenger og dreieledd.

James Watts kobling fra 1784 er et klassisk eksempel. Et punkt på koblingsstangen følger ikke en perfekt rett linje, men en del av banen kan være svært nær rett. Watts mer omfattende *parallel motion* brukte flere ledd for å føre stempelstangen i en dampmaskin.

Prosjektet har tre hoveddeler:

1. **Posisjonskinematikk:** rotasjoner, stive stenger og sirkel-skjæringer
2. **Hvor rett er banen?:** beste rette linje, egenverdier og egenvektorer
3. **Bevegelse langs banen:** numerisk hastighet og en gitt hastighetsmatrise

En valgfri avslutning bruker en enkel ODE for drivleddet. Studentene skal ikke utlede Jacobimatriser eller bruke vektoranalyse.

### Læringsmål

Etter prosjektet skal du kunne

- bruke en rotasjonsmatrise i planet,
- beregne posisjonen til en plan toleddet arm,
- kontrollere stanglengder med avstandsformelen,
- finne et skjæringspunkt mellom to sirkler,
- beregne en koblingskurve,
- bruke egenverdier og egenvektorer til å finne en best tilpasset linje,
- måle maksimal- og RMS-avvik fra en rett linje,
- estimere hastighet med sentrale differanser,
- løse et ferdig oppgitt lineært system for leddhastigheter,
- simulere en enkel ODE for et drivledd med Euler.

### Modellavgrensning

Modellen beskriver idealiserte, plane og stive stenger med perfekte dreieledd. Krefter, friksjon, slark, elastisitet og kollisjoner er utelatt i hovedmodellen.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Del A: Posisjonskinematikk

## A.1 En stang som roterer

En stang med lengde $L$ ligger først langs positiv $x$-akse. Når den roteres vinkelen $\theta$, blir endepunktsvektoren

$$
r(\theta)=R(\theta)
\begin{pmatrix}L\\0\end{pmatrix},
$$

med rotasjonsmatrisen

$$
\boxed{
R(\theta)=
\begin{pmatrix}
\cos\theta&-\sin\theta\\
\sin\theta&\cos\theta
\end{pmatrix}.}
$$

## Oppgave A1: Kontroller rotasjonsmatrisen

1. Implementer $R(\theta)$.
2. Kontroller numerisk at $R^TR=I$.
3. Kontroller at $\det R=1$.
4. Roter vektoren $(2,0)^T$ med $30^\circ$, $90^\circ$ og $180^\circ$.
5. Kontroller at vektorlengden bevares.

In [ ]:
def rotasjon(theta):
    return np.array([
        [..., ...],
        [..., ...]
    ], dtype=float)

for grader in [30, 90, 180]:
    theta = np.deg2rad(grader)
    R = rotasjon(theta)
    v = np.array([2.0, 0.0])
    v_rot = ...
    print(grader, "grader:", v_rot)
    print("R^T R =
", ...)
    print("det(R) =", ...)
    print("lengde =", ...)

## A.2 En åpen toleddet arm

En plan arm med to stenger har lengdene $L_1$ og $L_2$. Det første leddet har vinkel $\theta_1$, og det andre har relativ vinkel $\theta_2$.

Endepunktet er

$$
P=
L_1
\begin{pmatrix}
\cos\theta_1\\\sin\theta_1
\end{pmatrix}
+
L_2
\begin{pmatrix}
\cos(\theta_1+\theta_2)\\
\sin(\theta_1+\theta_2)
\end{pmatrix}.
$$

Dette er den samme grunnideen som brukes i plan robotkinematikk.

## Oppgave A2: Beregn og tegn armen

Bruk $L_1=1.5$, $L_2=1.0$, $\theta_1=35^\circ$ og $\theta_2=-70^\circ$. Beregn leddpunktet og endepunktet. Tegn stengene.

In [ ]:
L1 = 1.5
L2 = 1.0
theta1 = np.deg2rad(35.0)
theta2 = np.deg2rad(-70.0)

O = np.array([0.0, 0.0])
ledd = ...
P_arm = ...

plt.plot([O[0], ledd[0], P_arm[0]],
         [O[1], ledd[1], P_arm[1]], "o-")
plt.axis("equal")
plt.grid()
plt.show()

# A.3 Watt-koblingen som en lukket kjede

Vi bruker fire punkter:

- $A$ og $D$ er faste dreiepunkter,
- $B$ og $C$ er bevegelige ledd,
- $AB$ og $DC$ har lengde $L$,
- $BC$ har lengde $\ell$.

```text
A o---------o B
              \\
               o P
              /
D o---------o C
```

Når inngangsvinkelen $\theta$ er kjent, er

$$
B=A+L
\begin{pmatrix}
\cos\theta\\\sin\theta
\end{pmatrix}.
$$

Punktet $C$ må samtidig oppfylle

$$\|C-D\|=L$$

og

$$\|C-B\|=\ell.$$

Dermed er $C$ et skjæringspunkt mellom to sirkler.

## A.4 Skjæring mellom to sirkler

La sirklene ha sentre $S_0,S_1$, radier $r_0,r_1$ og senteravstand

$$d=\|S_1-S_0\|.$$

Avstanden fra $S_0$ til fotpunktet på senterlinjen er

$$
a=\frac{r_0^2-r_1^2+d^2}{2d}.
$$

Høyden fra senterlinjen til skjæringspunktene er

$$
h=\sqrt{r_0^2-a^2}.
$$

Enhetsvektoren langs senterlinjen er

$$e=\frac{S_1-S_0}{d},$$

og en vinkelrett enhetsvektor er

$$e_\perp=(-e_y,e_x)^T.$$

Skjæringspunktene er

$$Q\pm he_\perp,$$

hvor

$$Q=S_0+ae.$$

## Oppgave A3: Implementer sirkel-skjæringen

Funksjonen skal returnere to punkter. Dersom sirklene ikke skjærer hverandre, skal den returnere `None`.

In [ ]:
def sirkel_skjæring(S0, r0, S1, r1):
    S0 = np.asarray(S0, dtype=float)
    S1 = np.asarray(S1, dtype=float)
    diff = S1 - S0
    d = np.linalg.norm(diff)

    if d == 0 or d > r0 + r1 or d < abs(r0 - r1):
        return None

    a = ...
    h2 = ...
    if h2 < -1e-12:
        return None
    h = np.sqrt(max(h2, 0.0))

    e = ...
    e_perp = ...
    Q = ...

    return Q + h*e_perp, Q - h*e_perp

# Enkel kontroll:
print(sirkel_skjæring([0, 0], 1, [1, 0], 1))

## A.5 Velg riktig monteringsgren

To sirkler gir vanligvis to mulige punkter $C$. De svarer til to ulike monteringer av mekanismen.

I hovedoppgaven velger vi først punktet med størst $y$-koordinat. Når vi følger en hel bevegelse, velger vi deretter det punktet som ligger nærmest forrige verdi av $C$. Da unngår vi at mekanismen hopper uventet mellom de to grenene.

In [ ]:
d = 2.0
L = 1.40
ell = 1.00
alpha = 0.50

A = np.array([-d/2, 0.0])
D = np.array([ d/2, 0.0])


def watt_posisjon(theta, C_forrige=None, alpha=alpha):
    B = A + rotasjon(theta) @ np.array([L, 0.0])
    kandidater = sirkel_skjæring(D, L, B, ell)

    if kandidater is None:
        return None

    C1, C2 = kandidater
    if C_forrige is None:
        C = C1 if C1[1] >= C2[1] else C2
    else:
        C = C1 if np.linalg.norm(C1-C_forrige) <= np.linalg.norm(C2-C_forrige) else C2

    P = (1-alpha)*B + alpha*C
    return B, C, P

## Oppgave A4: Tegn én konfigurasjon

Velg en inngangsvinkel der koblingen kan monteres. Tegn $A,B,C,D$ og sporingspunktet $P$.

Kontroller numerisk de tre stanglengdene:

$$\|B-A\|=L,\qquad \|C-D\|=L,\qquad \|C-B\|=\ell.$$

In [ ]:
theta_test = np.deg2rad(80.0)
resultat = watt_posisjon(theta_test)
B, Cpunkt, P = resultat

print("Stanglengder:",
      np.linalg.norm(B-A),
      np.linalg.norm(Cpunkt-D),
      np.linalg.norm(Cpunkt-B))

plt.plot([A[0], B[0], Cpunkt[0], D[0]],
         [A[1], B[1], Cpunkt[1], D[1]], "o-")
plt.plot(P[0], P[1], "ro", label="sporingspunkt P")
plt.axis("equal")
plt.grid()
plt.legend()
plt.show()

# Del B: Hvor rett er banen?

## B.1 Beregn koblingskurven

Vi lar inngangsvinkelen variere gjennom et valgt intervall. For hvert vinkelpunkt beregner vi $B$, $C$ og $P$.

Ikke alle vinkler gir en mulig montering. Slike punkter hoppes over.

In [ ]:
theta_verdier = np.deg2rad(np.linspace(35.0, 145.0, 401))

P_liste = []
B_liste = []
C_liste = []
theta_gyldig = []
C_forrige = None

for theta in theta_verdier:
    resultat = watt_posisjon(theta, C_forrige=C_forrige)
    if resultat is None:
        continue
    B, Cpunkt, P = resultat
    B_liste.append(B)
    C_liste.append(Cpunkt)
    P_liste.append(P)
    theta_gyldig.append(theta)
    C_forrige = Cpunkt

P_bane = np.array(P_liste)
B_bane = np.array(B_liste)
C_bane = np.array(C_liste)
theta_gyldig = np.array(theta_gyldig)

plt.plot(P_bane[:, 0], P_bane[:, 1])
plt.axis("equal")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Koblingskurven til sporingspunktet")
plt.grid()
plt.show()

## B.2 Velg et nyttig arbeidsintervall

Bare en del av kurven er omtrent rett. Velg et sammenhengende vinkelintervall rundt den delen som ser mest rett ut.

I pilotverdiene kan du begynne med et intervall rundt midten av den beregnede banen. Senere kan intervallet justeres.

In [ ]:
# Eksempel: sentrale 45 prosent av de beregnede punktene.
n = len(P_bane)
i0 = int(0.275*n)
i1 = int(0.725*n)

P_arbeid = P_bane[i0:i1]
theta_arbeid = theta_gyldig[i0:i1]

plt.plot(P_bane[:, 0], P_bane[:, 1], color="lightgray", label="hele banen")
plt.plot(P_arbeid[:, 0], P_arbeid[:, 1], label="valgt arbeidsintervall")
plt.axis("equal")
plt.grid()
plt.legend()
plt.show()

## B.3 Beste rette linje med egenvektorer

La middelpunktet til banepunktene være

$$\overline P=\frac1n\sum_jP_j.$$

Sentrer punktene:

$$\widetilde P_j=P_j-\overline P.$$

Samle dem som rader i matrisen $X$. Da er

$$S=X^TX$$

en symmetrisk $2\times2$-matrise.

- Egenvektoren til den største egenverdien gir retningen langs banen.
- Egenvektoren til den minste egenverdien gir normalretningen til den best tilpassede linjen.

Dette er samme idé som å rotere koordinatsystemet slik at første akse ligger langs punktmengden.

## Oppgave B1: Finn hovedretning og normalretning

Beregn $S$, egenverdier og egenvektorer. Sorter egenvektorene etter stigende egenverdi.

In [ ]:
P_middel = ...
X = ...
S = ...

egenverdier, egenvektorer = ...
rekkefølge = np.argsort(egenverdier)
egenverdier = egenverdier[rekkefølge]
egenvektorer = egenvektorer[:, rekkefølge]

normal = egenvektorer[:, 0]
hovedretning = egenvektorer[:, 1]

print("Egenverdier:", egenverdier)
print("Normalretning:", normal)
print("Hovedretning:", hovedretning)
print("Kontroll ortogonalitet:", normal @ hovedretning)

## B.4 Mål retthetsfeilen

Sideavviket fra den best tilpassede linjen er

$$d_j=\widetilde P_j\cdot n,$$

hvor $n$ er normalretningen.

Vi bruker to mål:

$$
E_{maks}=\max_j|d_j|,
$$

$$
E_{RMS}=\sqrt{\frac1n\sum_jd_j^2}.
$$

Slaglengden langs linjen kan beregnes ved å projisere punktene på hovedretningen.

In [ ]:
sideavvik = ...
langskoordinat = ...

E_maks = ...
E_rms = ...
slaglengde = ...

print("Maksimalt sideavvik:", E_maks)
print("RMS-avvik:", E_rms)
print("Slaglengde:", slaglengde)
print("Relativ maksimal feil:", E_maks/slaglengde)

In [ ]:
s_min = np.min(langskoordinat)
s_max = np.max(langskoordinat)
linje = np.array([
    P_middel + s_min*hovedretning,
    P_middel + s_max*hovedretning
])

plt.plot(P_arbeid[:, 0], P_arbeid[:, 1], label="Watt-bane")
plt.plot(linje[:, 0], linje[:, 1], "--", label="beste rette linje")
plt.axis("equal")
plt.grid()
plt.legend()
plt.show()

## Oppgave B2: Parameterstudie

Undersøk virkningen av

- ytterstanglengden $L$,
- koblingslengden $\ell$,
- avstanden $d$ mellom de faste dreiepunktene,
- sporingspunktets plassering $\alpha$,
- størrelsen på arbeidsintervallet.

For hver variant registrerer du

- maksimal retthetsfeil,
- RMS-feil,
- slaglengde.

Diskuter kompromisset mellom lang slaglengde og liten sidefeil. Ikke bruk automatisk optimering i første omgang. Sammenlign noen få velvalgte geometrier.

# Del C: Bevegelse langs banen

## C.1 Numerisk hastighet uten vektoranalyse

Vi har allerede beregnet punktene

$$P(\theta_0),P(\theta_1),\ldots.$$

Den deriverte med hensyn på inngangsvinkelen kan estimeres med sentral differanse:

$$
\frac{dP}{d\theta}(\theta_j)
\approx
\frac{P_{j+1}-P_{j-1}}{\theta_{j+1}-\theta_{j-1}}.
$$

Hvis inngangsleddet roterer med kjent konstant vinkelhastighet

$$\dot\theta=\Omega,$$

gir vi sammenhengen

$$
\boxed{
\dot P\approx\frac{dP}{d\theta}\,\Omega.}
$$

Studentene skal bruke denne formelen, ikke utlede en generell kjerneregel for vektorfunksjoner.

## Oppgave C1: Fart langs arbeidsintervallet

Bruk $\Omega=1$ rad/s. Beregn hastighetsvektoren med sentrale differanser og deretter farten

$$v=\|\dot P\|.$$
$$

Undersøk om farten er omtrent konstant langs den rette delen.

In [ ]:
Omega_inn = 1.0

dP_dtheta = (P_arbeid[2:] - P_arbeid[:-2]) / (
    theta_arbeid[2:, None] - theta_arbeid[:-2, None]
)

hastighet = ...
fart = ...
theta_fart = theta_arbeid[1:-1]

plt.plot(np.rad2deg(theta_fart), fart)
plt.xlabel("Inngangsvinkel i grader")
plt.ylabel("Fart til P")
plt.grid()
plt.show()

print("Minste fart:", np.min(fart))
print("Største fart:", np.max(fart))

## Oppgave C2: Bevegelsesretning

Projiser hastighetsvektoren på

- hovedretningen,
- normalretningen.

Den normale komponenten måler hvor raskt punktet beveger seg sideveis bort fra den ønskede rette retningen.

In [ ]:
v_langs = ...
v_side = ...

plt.plot(np.rad2deg(theta_fart), v_langs, label="langs linjen")
plt.plot(np.rad2deg(theta_fart), v_side, label="sideveis")
plt.xlabel("Inngangsvinkel i grader")
plt.ylabel("Hastighetskomponent")
plt.grid()
plt.legend()
plt.show()

# C.2 En ferdig oppgitt hastighetsmatrise

I en alternativ vinkelbeskrivelse kan mekanismen ha de ukjente leddvinklene $\psi$ og $\varphi$. Når stanglengdene skal forbli konstante, får man et lineært system for leddhastighetene:

$$
\boxed{
J(\theta,\psi,\varphi)
\begin{pmatrix}
\dot\psi\\\dot\varphi
\end{pmatrix}
=
r(\theta,\psi,\varphi)\dot\theta.}
$$

I dette prosjektet er $J$ og $r$ **gitt**. Studentene skal ikke utlede dem med flervariabel kjerneregel.

For en vanlig firestangssløyfe kan en mulig ferdig form være

$$
J=
\begin{pmatrix}
-\ell\sin\psi&L\sin\varphi\\
\ell\cos\psi&-L\cos\varphi
\end{pmatrix},
$$

$$
r=
\begin{pmatrix}
L\sin\theta\\-L\cos\theta
\end{pmatrix}.
$$

Fortegnene avhenger av hvordan vinkler og sløyferetning er definert.

## Oppgave C3: Løs for leddhastighetene

Bruk et gitt sett vinkler og $\dot\theta$. Bygg matrisen, beregn determinanten og løs systemet med `np.linalg.solve`.

Undersøk hva som skjer med løsningen når $|\det J|$ blir liten. Dette kalles en nær-singulær konfigurasjon.

In [ ]:
theta_J = np.deg2rad(70.0)
psi_J = np.deg2rad(115.0)
phi_J = np.deg2rad(40.0)
theta_dot = 1.0

J = np.array([
    [-ell*np.sin(psi_J),  L*np.sin(phi_J)],
    [ ell*np.cos(psi_J), -L*np.cos(phi_J)]
])

r_hast = np.array([
    L*np.sin(theta_J),
    -L*np.cos(theta_J)
])

print("det(J) =", ...)
ledd_hastigheter = ...
print("psi_dot, phi_dot =", ledd_hastigheter)

# Valgfri del D: En ODE for drivleddet

Selve den stive koblingen behandles geometrisk. Vi kan likevel modellere motoren eller svinghjulet som driver inngangsleddet.

La

$$\omega=\dot\theta.$$

En enkel modell er

$$
\boxed{
\begin{aligned}
\dot\theta&=\omega,\\
J_m\dot\omega&=T_m(t)-c\omega-k\sin\theta.
\end{aligned}}
$$

Dette er et vanlig ODE-system for drivleddet. For hvert beregnet $\theta(t)$ finner vi resten av koblingen med sirkelgeometrien fra del A.

## Oppgave D1: Euler for drivleddet

Bruk et konstant motormoment i en begrenset tidsperiode og deretter null moment. Simuler $\theta(t)$ og $\omega(t)$. Beregn sporingspunktet $P(t)$ for alle gyldige konfigurasjoner.

In [ ]:
J_m = 0.20
c_m = 0.08
k_m = 0.15
T0 = 0.25


def motor_moment(t):
    return T0 if t < 2.0 else 0.0


def drivledd(t, X):
    theta, omega = X
    dtheta = omega
    domega = ...
    return np.array([dtheta, domega])


def euler_system(f, X0, sluttid, h):
    n = int(round(sluttid/h))
    t = np.linspace(0.0, n*h, n + 1)
    X = np.zeros((n + 1, len(X0)))
    X[0] = X0

    for j in range(n):
        X[j + 1] = ...

    return t, X

# Simuler og beregn P(t) der geometrien er gyldig.

# Historisk fordypning

Watts mekanisme gir tilnærmet rettlinjet bevegelse med få ledd. Senere ble det utviklet mekanismer som gir eksakt rettlinjet bevegelse, blant annet Peaucellier–Lipkin-mekanismen.

Diskuter:

- Hvorfor kunne en god tilnærming være mer nyttig enn en komplisert eksakt løsning?
- Hvordan påvirker antall ledd produksjon, slark og vedlikehold?
- Hva er forskjellen mellom en åpen robotarm og en lukket koblingskjede?
- Hvorfor er singulariteter viktige både i historiske mekanismer og moderne roboter?

Den fullstendige Watt *parallel motion* kan studeres som en utvidelse der flere ledd flytter eller forstørrer den omtrent rette bevegelsen fra den enkle Watt-koblingen.

# Modellkritikk

Diskuter minst fem punkter:

- Stengene antas helt stive.
- Leddene antas uten friksjon og slark.
- Mekanismen er helt plan.
- Sirkel-skjæringen kan gi to monteringsgrener.
- Mekanismen kan ikke passere gjennom alle vinkler.
- En numerisk grenregel kan feile nær en singularitet.
- Den beste rette linjen avhenger av valgt arbeidsintervall.
- Numerisk hastighet avhenger av vinkelsteget.
- Hovedmodellen beregner ikke krefter eller leddmomenter.
- ODE-en i del D beskriver bare drivleddet, ikke full koblingsdynamikk.
- En virkelig mekanisme kan kollidere med seg selv eller omgivelsene.

# Oppsummering

Skriv en kort rapport der du forklarer

1. hvordan rotasjonsmatrisen brukes til å plassere en stang,
2. hvordan sirkel-skjæringen lukker Watt-koblingen,
3. hvordan du kontrollerte stanglengdene,
4. hvilken del av koblingskurven som var omtrent rett,
5. hvordan egenvektorene til $X^TX$ ga hovedretning og normalretning,
6. hvordan maksimal- og RMS-feil målte rettheten,
7. hvordan geometrien påvirket feil og slaglengde,
8. hvordan hastighet ble estimert uten vektoranalyse,
9. hva en liten determinant til hastighetsmatrisen betyr,
10. hvorfor Watts løsning var en tilnærmet, men ingeniørmessig nyttig rettlinjemekanisme.

## Referanser for videre lesning

- James Watts firestangskobling og den mer omfattende *parallel motion*-mekanismen.
- Historien om tilnærmede og eksakte rettlinjemekanismer.

Studentene trenger ikke lese eksterne kilder for å gjennomføre prosjektet.